In [ ]:
import cv2
import tensorflow as tf
import numpy as np

model = tf.keras.models.load_model("model.h5")
labels = np.load("labels.npy")

In [ ]:
#Opening camera

video = cv2.VideoCapture(0)

baseline_frame = None

while True:
    detected_input, video_frame = video.read()
    if not detected_input :
        print("No input detected!")
        break

    grayscale = cv2.cvtColor(video_frame, cv2.COLOR_BGR2GRAY)
    grayscale = cv2.GaussianBlur(grayscale, (21,21), 0)

    if baseline_frame is None:
        baseline_frame = grayscale
        continue

    frame_delta = cv2.absdiff(baseline_frame, grayscale)

    zero, threshold_frame = cv2.threshold(frame_delta, 25, 255, cv2.THRESH_BINARY)

    threshold_frame = cv2.dilate(threshold_frame, None, iterations = 2)

    contours, zero = cv2.findContours(threshold_frame.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    motion_sensor = False

    for contour in contours:
        if cv2.contourArea(contour) < 5000:
            continue

        motion_sensor = True

        (x ,y, w, h) = cv2.boundingRect(contour)

        cv2.rectangle(video_frame, (x,y), (x+w, y+h), (0,255,0), 2) 

    if motion_sensor == True :
        cv2.putText(video_frame, "STATUS : MOTION DETECTED", (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)   
    else:
        cv2.putText(video_frame, "STATUS : IDLE", (10,20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
        
    cv2.imshow("Security feed (Original)", video_frame)
    #cv2.imshow("Threshold Frame (Motion Binary)", threshold_frame)

    cv2.putText(
    video_frame,f"{action} ({score:.2f})",(10,30),cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()